In [0]:
# ═══════════════════════════════════════════════
# 01_INGESTION — Pull data from ONS API into Bronze
# ═══════════════════════════════════════════════
import pandas as pd
import requests
from io import StringIO

# 1. Get the latest version link from the ONS API
base_url = "https://api.beta.ons.gov.uk/v1/datasets/uk-spending-on-cards"
meta = requests.get(base_url).json()
latest_version_url = meta["links"]["latest_version"]["href"]
print("Latest version URL:", latest_version_url)

# 2. Convert the API version URL into the CSV download URL
#    API:      https://api.beta.ons.gov.uk/v1/datasets/.../versions/130
#    Download: https://download.ons.gov.uk/downloads/datasets/.../versions/130.csv
csv_url = latest_version_url.replace(
    "https://api.beta.ons.gov.uk/v1/datasets",
    "https://download.ons.gov.uk/downloads/datasets"
) + ".csv"
print("CSV download URL:", csv_url)

# 3. Download the CSV (User-Agent header avoids the 403 block)
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(csv_url, headers=headers)
response.raise_for_status()
pdf = pd.read_csv(StringIO(response.text))
print("Downloaded shape:", pdf.shape)

# 4. Convert to Spark and clean column names (Delta rejects spaces/hyphens)
bronze_df = spark.createDataFrame(pdf)
for old_col in bronze_df.columns:
    new_col = old_col.replace(" ", "_").replace("-", "_")
    bronze_df = bronze_df.withColumnRenamed(old_col, new_col)

# 5. Write to Bronze
bronze_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_card_spending")
print("Bronze table written. Rows:", spark.table("bronze_card_spending").count())

Latest version URL: https://api.beta.ons.gov.uk/v1/datasets/uk-spending-on-cards/editions/time-series/versions/130
CSV download URL: https://download.ons.gov.uk/downloads/datasets/uk-spending-on-cards/editions/time-series/versions/130.csv
Downloaded shape: (9075, 10)
Bronze table written. Rows: 9075
